# overlay_rnd: commented notebook version

This notebook mirrors the logic in `overlay_rnd/overlay_rnd.py`. It shows both how to drive the existing soft IOC and how to reproduce the same random-overlay logic directly in notebook code.

In [ ]:
import random
import time
from p4p.client.thread import Context

ctx = Context("pva")
CAM = "LAB:SIM:CAM01"
OVLIOC = "LAB:SIM:OVERLAY_RND"

def unwrap_scalar(value):
    if hasattr(value, "value"):
        value = value.value
    if isinstance(value, dict) and "value" in value:
        value = value["value"]
    return value

def get_scalar(pv):
    return unwrap_scalar(ctx.get(pv, timeout=2.0))

def put_scalar(pv, value, wait=True):
    ctx.put(pv, value, wait=wait, timeout=5.0)

def read_overlay_rect(camera_prefix):
    base = f"{camera_prefix}:Overlay1:1"
    return {
        "x": int(get_scalar(f"{base}:PositionX_RBV")),
        "y": int(get_scalar(f"{base}:PositionY_RBV")),
        "sx": int(get_scalar(f"{base}:SizeX_RBV")),
        "sy": int(get_scalar(f"{base}:SizeY_RBV")),
    }

In [ ]:
# Drive the existing overlay soft IOC through its control PVs.
for field, value in {"maxx": 250, "maxy": 180, "size": 80, "X": 850, "Y": 500}.items():
    put_scalar(f"{OVLIOC}:{field}", value)

put_scalar(f"{OVLIOC}:run", 1)
time.sleep(0.5)

{
    "outX": float(get_scalar(f"{OVLIOC}:outX")),
    "outY": float(get_scalar(f"{OVLIOC}:outY")),
    "camera_overlay": read_overlay_rect(CAM),
}

In [ ]:
def apply_overlay(camera_prefix, center_x, center_y, width, height):
    base = f"{camera_prefix}:Overlay1"
    pos_x = max(0, int(center_x - width // 2))
    pos_y = max(0, int(center_y - height // 2))
    put_scalar(f"{base}:EnableCallbacks", 1)
    put_scalar(f"{base}:1:Use", 1)
    put_scalar(f"{base}:1:Shape", 1)
    put_scalar(f"{base}:1:PositionX", pos_x)
    put_scalar(f"{base}:1:PositionY", pos_y)
    put_scalar(f"{base}:1:SizeX", int(width))
    put_scalar(f"{base}:1:SizeY", int(height))
    return {"center": (int(center_x), int(center_y)), "position": (pos_x, pos_y), "size": (int(width), int(height))}

def random_overlay(camera_prefix, base_x=900, base_y=500, maxx=200, maxy=200, size=60):
    center_x = base_x + random.randint(-int(maxx), int(maxx))
    center_y = base_y + random.randint(-int(maxy), int(maxy))
    return apply_overlay(camera_prefix, center_x, center_y, size, size)

In [ ]:
# Notebook-native version of the same random-overlay logic.
history = []
for _ in range(5):
    history.append(random_overlay(CAM, base_x=900, base_y=500, maxx=180, maxy=120, size=60))
    time.sleep(0.4)

history